# AI for testing and calibration {#sec-ai-test}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/book-ai/notebooks/c20-ai-test.ipynb)

The waiting-list model has passed chapter 12's test sheet and been calibrated in chapter 13. Now a colleague sends a "simpler" rewrite, the board wants a forecast, and a data scientist asks why not let a learned model do it all. AI can help with each job: a reasoning model can plan tests, an agent can hunt for failures or choose what to calibrate, and machine learning can learn one effect or set a forecasting benchmark. Each of them can also be wrong in a fluent, confident way. This chapter puts every AI step through a check by code before anything rests on it.

```
            reasoning model: a test plan (1)     agent: hunts for failures (2)
                         \                         /
                          v                       v
 Referrals ----> [ Waiting list, four bands ] ----> Treatments
                          ^                         = MIN(Capacity, list / 0.5)
                          |                              ^
      Capacity: levels and recovery chosen by     the limit a rewrite
      a calibration agent (5), judged on          drops (2), and an effect
      2022 to 2023, which it never saw            learned from data (3)
```

*The model under test and where AI acts on it: a reasoning model plans tests, agents hunt for failures and choose parameters to fit, and machine learning learns one effect. Numbers in brackets are sections.*

The chapter follows the tasks of the test stage: plan the tests (section 1), hunt for failures (section 2), learn a part of the structure from data (section 3), compete with a data model across a policy change (section 4), calibrate (section 5), and avoid the traps of fitting (section 6). A **Failure gallery** then shows the ways AI goes wrong at this stage, each caught by a check. @sec-ai-partners introduced language models, reasoning models, agents, the replay files and the contract for using them; this chapter uses them without explaining them again.

**Who it is for.** Modellers who want AI help with testing and calibration without handing over their judgement, and data scientists who want to see where learned models fit in a feedback model. **Who can skip.** Nobody should skip section 4 or the Failure gallery; readers who know hybrid models may skim section 3. **Prerequisites.** @sec-ai-partners for the AI tools, @sec-learning for training, testing and overfitting, @sec-testing for the tests and @sec-calibration for calibration. **Time.** About three hours.

**Where the replies come from.** No cell in this chapter calls a model service. Every reply was written for the book, with the help of the model named in the setup cell, and recorded in `data/ai/c20.json`; each record says so in its `how` field. Some replies contain faults planted on purpose, and the text says which. To ask a live model instead, run `pip install anthropic`, set `ANTHROPIC_API_KEY` to your key and `LB_AI_LIVE=1`, and run the chapter: requests that have a recorded reply still replay, and new ones go to the live model and are recorded as live calls.

**Learning outcomes.** By the end of this chapter you can:

1. **design** a test plan with a reasoning model and **map** each proposed test to code that runs it (section 1, Try it 1);
2. **verify** a testing agent's claimed failure by rerunning it, and **check** every number in its report against the tool results (section 2, Try it 2);
3. **build** a hybrid model whose nonlinear effect is learned from data inside a stock-and-flow structure (section 3, Try it 3);
4. **evaluate** a pure data model against a system dynamics model across a policy change, and **explain** the difference (section 4, Try it 4);
5. **supervise** a calibration agent and **judge** its choice on held-out years that none of its tools could read (section 5, Try it 5);
6. **diagnose** overfitting and leakage in a forecasting experiment (section 6, Try it 6);
7. **catch** each failure in the gallery with a check by code (Failure gallery, Try it 7).

In [ ]:
# Run this cell first. In Colab it installs loopbricks (about ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@book-ai#subdirectory=python"])
    import loopbricks as lb
import os, re, json
from functools import lru_cache       # remembers results of slow calls
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from loopbricks import ai

SEED = 20
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/book-ai/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])
# the recorded replies of every model call in this chapter (see section 1)
REPLAY = "data/ai/c20.json" if os.path.exists("data/ai/c20.json") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/book-ai/notebooks/data/ai/c20.json"

# Okabe-Ito colours, each with its own line style, so figures read in
# greyscale.
# rcParams holds matplotlib's defaults; cycler gives each new line the next
# colour and style together.
OKABE_ITO = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00",
             "#0072B2", "#CC79A7"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)),
          (0, (1, 3))]
plt.rcParams["axes.prop_cycle"] = cycler(color=OKABE_ITO) + \
    cycler(linestyle=STYLES)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__, "| data:", len(wl), "months | model:",
      ai.MODEL)

## 1. Reasoning: a test plan, mapped to tests that run

*Stage task:* decide which tests to run before the model is used to argue for capacity. *Kind of AI:* a reasoning model (`reasoning=True`), which thinks before it answers [@wei2022cot]. *Check:* code maps every proposed test to a function that runs it, and the run, not the model, decides pass or fail. *What stays human:* the tests that no function can run, and the judgement of whether the plan is enough.

The model under test is chapter 12's: the list in four bands of time waited, with treatments shared by size and a `Priority` weight on the newest band. We rebuild it in one function, with the values set by eye in chapter 12.

In [ ]:
BANDS = ["Waiting 0 to 3", "Waiting 3 to 6", "Waiting 6 to 12",
         "Waiting 12 plus"]
KEYS = [b.replace(" ", "_") for b in BANDS]       # names in equations
first = wl.iloc[0]


def waiting_list_spec():
    """Chapter 12's model: the list in four bands of time waited."""
    def var(name, eq, units):
        return {"id": lb.name_key(name), "name": name, "eq": str(eq),
                "units": units}

    def flow(name, eq, frm=None, to=None):
        f = {"id": lb.name_key(name), "name": name, "eq": eq,
             "units": "patients/month"}
        f.update({k: v for k, v in (("from", frm), ("to", to)) if v})
        return f

    stocks = [{"id": f"b{i}", "name": b, "init": str(first[c]),
               "units": "patients", "nonNegative": True}
              for i, (b, c) in enumerate(zip(
                  BANDS, ["w0_3", "w3_6", "w6_12", "w12_plus"]))]
    flows = [flow("Referrals", "Referrals_0 * EXP(Referral_growth * TIME"
                  " / Months_per_year)", to="b0")]
    flows += [flow(f"Ageing {i}", f"{KEYS[i]} / Band_time_{i}", f"b{i}",
                   f"b{i + 1}") for i in range(3)]
    flows += [flow(f"Treating {i}", "Treatments * " + ("Priority * "
                   if i == 0 else "") + f"{KEYS[i]} / Weighted_list",
                   f"b{i}") for i in range(4)]
    weighted = f"Priority * {KEYS[0]} + " + " + ".join(KEYS[1:])
    variables = [
        var("Waiting list", " + ".join(KEYS), "patients"),
        var("Weighted list", f"MAX({weighted}, Smallest_list)", "patients"),
        var("Treatments",
            "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
            "patients/month"),
        var("Capacity", "Capacity_0 * EXP(Capacity_growth * TIME"
            " / Months_per_year) * Shock_effect"
            " * (1 + STEP(Initiative_boost, Initiative_start))",
            "patients/month"),
        var("Shock effect", "1 - STEP(1, Shock_start)"
            " * (1 - Activity_at_worst)"
            " * MAX(0, 1 - (TIME - Shock_start) / Recovery_time)", "dmnl")]
    variables += [var(n, v, u) for n, v, u in [
        ("Referrals 0", 4000, "patients/month"),
        ("Referral growth", 0.03, "1/year"),
        ("Capacity 0", 3950, "patients/month"),
        ("Capacity growth", 0.02, "1/year"),
        ("Activity at worst", 0.35, "dmnl"), ("Recovery time", 9, "month"),
        ("Shock start", 73, "month"), ("Initiative boost", 0.10, "dmnl"),
        ("Initiative start", 95, "month"), ("Priority", 1.4, "dmnl"),
        ("Minimum treatment time", 0.5, "month"),
        ("Months per year", 12, "month/year"),
        ("Band time 0", 3, "month"), ("Band time 1", 3, "month"),
        ("Band time 2", 6, "month"), ("Smallest list", 1, "patients")]]
    return {"name": "Waiting list by band", "timeUnit": "month", "T": 119,
            "dt": 0.25, "stocks": stocks, "flows": flows, "vars": variables}


case = lb.Model.from_spec(waiting_list_spec())
base = case.run(include=["Waiting list"])
print(f"December 2023: model {base['Waiting list'].iloc[-1]:,.0f}, "
      f"data {wl.total.iloc[-1]:,} patients")

The model's December 2023 list, 68,836 patients against 69,927 in the data, matches chapter 12.

A plan from a model is only useful if it can be run. So the prompt gives the model a fixed menu of checks, each one a Python function we write below, and asks for the plan as JSON. Anything the menu cannot do, the model must mark `person`. The prompt also asks for tests that could fail, because a plan of tests that are sure to pass proves nothing (the Failure gallery shows one).

In [ ]:
PLAUSIBLE = {"referrals_0": (3000, 5000), "referral_growth": (0.0, 0.05),
             "capacity_0": (3000, 5000), "capacity_growth": (-0.02, 0.05),
             "priority": (1.0, 3.0), "minimum_treatment_time": (0.25, 1.0)}
CHECKS = """never_negative: run with the settings; passes if no band goes
  below zero and the floor on the stocks never has to act.
grows_by_referrals: passes if the list rises by exactly the referrals
  added up (use with capacity_0 = 0).
empties: passes if the list ends below one patient without going negative.
conserved: passes if every stock equals its flows added up.
dt_stable: passes if halving dt moves the December 2023 list by under 1%.
history_10pct: passes if every month is within 10% of the data.
steepest_march_2020: passes if the steepest monthly rise is in March 2020.
knockout: replaces one equation ("equation": {"name": ..., "eq": ...})
  and reports how far the December 2023 list moves; give "expect_change"
  as "none" (under 1%) or "large" (1% or more).
person: anything the checks above cannot do; say who should do it."""

PLAN_PROMPT = f"""Here is a system dynamics model of an outpatient waiting
list, monthly from January 2014 (TIME 0) to December 2023 (TIME 119).

{case.equations()}

Plausible ranges of the main parameters: {PLAUSIBLE}

Propose a test plan for this model, before it is used to argue for more
capacity. Think about which tests could fail, not only which would pass.
Each test uses one of these checks, with "settings" as parameter values:
{CHECKS}

Return a JSON list; each test is an object with keys "name", "check",
"settings" (a dict, may be empty), "why" (one sentence) and, for a
knockout, "equation" and "expect_change". At most 12 tests."""
SYSTEM_TESTER = ("You are an experienced system dynamics modeller who "
                 "designs tests that could prove a model wrong.")
plan = ai.ask_json(PLAN_PROMPT, system=SYSTEM_TESTER, reasoning=True,
                   replay=REPLAY)
# the replay file holds each reply with a note of how it was made
records = json.load(open(REPLAY)) if os.path.exists(REPLAY) else \
    pd.read_json(REPLAY, typ="series").to_dict()
key = ai.request_key([{"role": "user", "content": PLAN_PROMPT}],
                     SYSTEM_TESTER, reasoning=True)
print("reply", key, "|", records[key]["how"], "|", records[key]["date"])
print(len(plan), "tests proposed")
for t in plan:
    print(f"- {t['name']} [{t['check']}] {t.get('settings', {})}")

The reasoning model proposed 12 tests. Its reply opens with a short explanation and then the JSON, which `ask_json` reads; the first line printed shows the record's key and how it was made. Its reasoning is worth reading: it expects the urgent-priority and shortest-treatment-time cases to fail, because the `MIN` in `Treatments` protects the whole list, not each band, and because `dt` equals the minimum treatment time at the low end of its range.

Those are hypotheses. The code below runs them. `run_test` looks up each test's check in `RUNNERS`, a dictionary from names to functions, and refuses any setting that is not a parameter of the model. Each runner returns "pass" or "FAIL" with the evidence. The model never decides a result.

In [ ]:
def balance_gap(model, settings):
    """Largest gap (patients) between any stock and its flows added up."""
    df = model.run(include="all", warn_clips=False, **settings)
    gaps = []
    for s in model.stocks:
        ins = [f["name"] for f in model.flows if f.get("to") == s["id"]]
        outs = [f["name"] for f in model.flows if f.get("from") == s["id"]]
        net = df[ins].sum(axis=1) - df[outs].sum(axis=1)
        added = df[s["name"]].iloc[0] + model.dt * net.cumsum().shift(
            fill_value=0)
        gaps.append((df[s["name"]] - added).abs().max())
    return max(gaps), df.attrs["clipped"]


def end_list(model, **kw):
    return model.run(include=["Waiting list"], warn_clips=False,
                     **kw)["Waiting list"].iloc[-1]


def run_test(model, test):
    """Run one test of a plan: ("pass" | "FAIL" | "for a person", detail)."""
    kind, s = test.get("check"), dict(test.get("settings") or {})
    if kind == "person":
        return "for a person", "the plan says a person should do this"
    if kind not in RUNNERS:
        return "for a person", f"no runner for the check {kind!r}"
    unknown = [k for k in s if k not in model.params]
    if unknown:
        return "for a person", f"not parameters of the model: {unknown}"
    return RUNNERS[kind](model, s, test)


def never_negative(model, s, test):
    df = model.run(warn_clips=False, **s)
    low, held = df[BANDS].min().min(), sorted(df.attrs["clipped"])
    ok = low >= 0 and not held
    return ("pass" if ok else "FAIL"), f"lowest band {low:,.0f}; floor " \
        f"held {held or 'nothing'}"


def grows_by_referrals(model, s, test):
    df = model.run(include=["Waiting list", "Referrals"], **s)
    rise = df["Waiting list"].iloc[-1] - df["Waiting list"].iloc[0]
    added = model.dt * df["Referrals"].iloc[:-1].sum()
    return ("pass" if abs(rise - added) < 1 else "FAIL"), \
        f"rise {rise:,.0f}, referrals {added:,.0f}"


def empties(model, s, test):
    df = model.run(warn_clips=False, **s)
    end, low = df["Waiting list"].iloc[-1], df[BANDS].min().min()
    ok = end < 1 and low >= 0 and not df.attrs["clipped"]
    return ("pass" if ok else "FAIL"), f"ends at {end:,.1f}, lowest band " \
        f"{low:,.1f}"


def conserved(model, s, test):
    gap, held = balance_gap(model, s)
    ok = gap < 1e-6 and not held
    return ("pass" if ok else "FAIL"), f"largest gap {gap:.1e}; floor " \
        f"held {sorted(held) or 'nothing'}"


def dt_stable(model, s, test):
    a, b = end_list(model, dt=0.25, **s), end_list(model, dt=0.125, **s)
    change = abs(a / b - 1)
    return ("pass" if change < 0.01 else "FAIL"), f"{change:.2%} change"


def months_of(model, s):
    run = model.run(include=["Waiting list"], warn_clips=False, **s)
    return run["Waiting list"].to_numpy()[::4]       # dt 0.25: every 4th row


def history_10pct(model, s, test):
    worst = np.abs(months_of(model, s) / wl.total.to_numpy() - 1).max()
    return ("pass" if worst < 0.10 else "FAIL"), f"worst month {worst:.1%}"


def steepest_march_2020(model, s, test):
    i = int(np.argmax(np.diff(months_of(model, s)))) + 1
    when = wl.month[i].strftime("%b %Y")
    return ("pass" if when == "Mar 2020" else "FAIL"), f"steepest {when}"


def knockout(model, s, test):
    eq = test["equation"]
    a = end_list(model, **s)
    b = end_list(model.with_equation(eq["name"], eq["eq"]), **s)
    moved = abs(b / a - 1)
    ok = (moved < 0.01) == (test.get("expect_change") == "none")
    return ("pass" if ok else "FAIL"), f"list moves {moved:.2%}"


RUNNERS = {f.__name__: f for f in [never_negative, grows_by_referrals,
                                   empties, conserved, dt_stable,
                                   history_10pct, steepest_march_2020,
                                   knockout]}


def run_plan(model, plan):
    rows = [(t["name"], t["check"], *run_test(model, t)) for t in plan]
    return pd.DataFrame(rows, columns=["test", "check", "result", "detail"])


pd.set_option("display.max_colwidth", 60)
sheet = run_plan(case, plan)
print(sheet.to_string(index=False))
print(sheet.result.value_counts().to_dict())

Ten tests ran and passed; two went to a person. With no capacity the list rises by 554,208 patients, exactly the referrals added up. Capacity at the top of its range and at ten times the base leaves the lowest band at 7 patients, and the floor never acts. Patients are conserved to $1.8 \times 10^{-11}$, the worst month is 4.4% off the data, and halving `dt` with three times the capacity changes the list by 0.00%.

The two predicted failures did not happen: the urgent-priority case leaves 28 patients in the lowest band, and the shortest treatment time leaves 1. The reasoning was plausible and wrong at these settings; chapter 12 needed a priority of 50, far outside the range, to overdraw a band. A reasoning model's "why" is a hypothesis, and the run decides.

The knockout is the most useful result. Replacing `Treatments` by plain `Capacity` moves the December 2023 list by 0.00%, so the limit `Waiting_list / Minimum_treatment_time` never acts in the history. It matters only when capacity exceeds demand. No behaviour test can tell whether that limit is there, which is exactly the opening a careless rewrite needs (section 2).

Two tests went to a person, for two different reasons. The empty-start test named the stocks as if they were parameters; the check caught it, because initial values are set with `with_init`, not with settings. The boundary test, referrals that respond to the wait, needs new structure, and the plan said so. Chapter 12 showed that this one matters for the policy conclusion. That is the modeller's work, with a GP lead.

**Try it 1.**

- *Check.* How many of the plan's tests did the code run, and how many went to a person? Which of the model's predicted failures happened?
- *Apply.* Write a runner `empty_start` that empties every band with `with_init` and runs with the test's settings, add it to `RUNNERS`, and run the empty-list test with no referrals.
- *Extend.* Plant a typo: `Ageing 2` divided by `Band_time_1` (3 months) instead of `Band_time_2` (6 months). Run the whole plan on it. Which tests catch it, and what does that say about the plan?

<details><summary>Worked answer</summary>

*Check.* Ten ran (all passed) and two went to a person. None of the predicted failures happened at the settings the plan chose.

*Apply.* The next cell defines `empty_start`. The guarded model passes: the largest list is 0.0, with no crash and no clip. The guard `MAX(..., Smallest_list)` in `Weighted list` is doing its job.

*Extend.* No test catches it. The typo moves patients into the 12-plus band twice as fast, so the share waiting over 12 months in December 2023 rises from 47% to 55%, but the total stays at 68,836. Every behaviour test in the plan looks at the total. A plan needs a test of the bands, such as chapter 12's share over 12 months against the data's 50%. Planting a fault and asking which tests catch it is called **mutation testing**; it measures a plan's power, not just its passes.

</details>

In [ ]:
def empty_start(model, s, test):
    """Every band empty at the start, then run with the settings."""
    for b in BANDS:
        model = model.with_init(b, 0)
    try:
        df = model.run(warn_clips=False, **s)
    except lb.LoopBricksError as err:     # the error loopbricks raises
        return "FAIL", f"crashed: {str(err)[:50]}"
    ok = df["Waiting list"].max() == 0 and not df.attrs["clipped"]
    return ("pass" if ok else "FAIL"), \
        f"largest list {df['Waiting list'].max():,.1f}"


RUNNERS["empty_start"] = empty_start
print("check:", sheet.result.value_counts().to_dict())
print("apply:", run_test(case, {"check": "empty_start",
                                "settings": {"referrals_0": 0}}))
typo = case.with_equation("Ageing 2", "Waiting_6_to_12 / Band_time_1")
caught = run_plan(typo, plan)
print("extend: tests that catch the typo:",
      list(caught.test[caught.result == "FAIL"]) or "none")
for label, m in [("case", case), ("typo", typo)]:
    d = m.run(include=BANDS + ["Waiting list"]).iloc[-1]
    print(f"extend: {label}: list {d['Waiting list']:,.0f}, over 12 months "
          f"{d[BANDS[3]] / d['Waiting list']:.0%}")

## 2. Agent: hunting for failures in a rewrite

*Stage task:* find the conditions under which a revised model fails. *Kind of AI:* an agent that calls tools in a loop, choosing each test from the last result [@yao2022react]. *Check:* code reruns the failure the agent claims and compares every number in its report with what the tools returned. *What stays human:* the fix, and the decision that the rewrite is or is not acceptable.

A colleague has sent a "simpler" version of the model. Two things changed. The stocks have no floor, so a stock that should go negative shows it, instead of being held at zero; that is good practice for testing. And `Treatments` is now plain `Capacity`, the knockout of section 1. The agent is not told about the second change.

In [ ]:
rewrite = waiting_list_spec()
for s in rewrite["stocks"]:
    s["nonNegative"] = False                  # let errors show, not hide
variant = lb.Model.from_spec(rewrite).with_equation("Treatments",
                                                    "Capacity")
gap = (variant.run(include=["Waiting list"])["Waiting list"]
       - base["Waiting list"]).abs().max()
print(f"largest difference from the case model, base run: {gap:.1e} "
      "patients")

In the base run the rewrite and the case model agree to 0.0 patients, so every behaviour test of chapter 12 passes it. Only a test outside the history can tell them apart.

The agent's tools are ordinary functions that summarise a run of the rewrite. Each docstring is what the agent reads about the tool, so it says what goes in and what comes out. The tools round their results for reading, and they compute every number the agent will see.

In [ ]:
def summary(model, settings, dt=None):
    """The facts a tester needs from one run, rounded for reading."""
    df = model.run(include=BANDS + ["Waiting list"], warn_clips=False,
                   **({"dt": dt} if dt else {}), **settings)
    below = df.index[(df[BANDS] < 0).any(axis=1)]
    return {"list at start": round(df["Waiting list"].iloc[0]),
            "list at end": round(df["Waiting list"].iloc[-1]),
            "lowest list": round(df["Waiting list"].min()),
            "lowest band": round(df[BANDS].min().min()),
            "first month a band is negative":
                float(below[0]) if len(below) else None}


def equations() -> str:
    """The model's equations, as text."""
    return variant.equations()


def simulate(settings: dict) -> dict:
    """Run the model with parameter settings ({} for the base run). Returns
    the list (patients) at the start and end, its lowest value, the lowest
    band, and the first month any band is below zero."""
    return summary(variant, settings)


def extreme(parameter: str, value: float) -> dict:
    """Run with one parameter pushed to a value; the same facts as
    simulate."""
    return {"parameter": parameter, "value": value,
            **summary(variant, {parameter: value})}


def halve_dt(settings: dict) -> dict:
    """The same run at dt 0.25, 0.125 and 0.0625 months: the list at the end
    and the first month a band is negative, for each dt."""
    return {str(d): {k: v for k, v in summary(variant, settings, d).items()
                     if k in ("list at end", "first month a band is negative")}
            for d in (0.25, 0.125, 0.0625)}


def mass_balance(settings: dict) -> dict:
    """Largest gap (patients) between any stock and its flows added up, and
    the stocks the floor had to hold (there is no floor in this model)."""
    gap, held = balance_gap(variant, settings)
    return {"largest gap": float(f"{gap:.1e}"), "floor held": sorted(held)}


TEST_TOOLS = {f.__name__: f for f in [equations, simulate, extreme,
                                      halve_dt, mass_balance]}
print(simulate({}))

The task asks for the clearest failure as a tool call that can be rerun, and a report that quotes only numbers the tools returned.

In [ ]:
HUNT = f"""A colleague revised our waiting-list model and turned off the
floor on its stocks, so that errors show instead of being hidden. Hunt for
conditions under which the revised model fails: impossible values, results
that depend on the time step, patients created or lost. Plausible ranges:
{PLAUSIBLE}. Test inside the ranges, at their ends and beyond.
Finish with an answer that is a JSON object: {{"failure": {{"tool": ...,
"args": ...}} for the clearest failure you found, or null, and "report":
three sentences for the modeller that quote only numbers the tools
returned}}."""
hunt = ai.agent(HUNT, TEST_TOOLS, replay=REPLAY, max_steps=10)
ai.show(hunt, width=110)

The replay shows seven steps; `ai.show` prints each thought, the tool it called and the start of what came back. The agent read the equations first and saw that `Treatments` no longer looks at the list. It ran the base case, then pushed capacity above referrals. At `capacity_0` = 5,000, the top of its plausible range, a band is negative from month 33.25 and the lowest band reaches minus 75,856 patients. It then ruled out the time step: the first negative month is 33.25 at every `dt`. It checked the books, which balance to $3.1 \times 10^{-11}$: the arithmetic faithfully conserves impossible negative patients. Last, it broke the model from the other side, with referrals at 3,000.

Halving `dt` did change one thing: the list at the end is 369,194, 28,609 or 22,739 patients, depending on the step. A model that has left the region where it means anything often behaves like this, and the agent said so.

Now the checks. An agent's report is a claim, so we rerun the claimed failure and compare. `numbers_in` reads every number in a text with a **regular expression**, a pattern for text, and `unsourced` lists the numbers that no source holds, at the precision quoted. The sources are the task, the agent's own tool arguments and every tool result.

In [ ]:
NUMBER = r"(?<![\w.])-?\d[\d,]*(?:\.\d+)?"     # 68,836 or -3.5 or 0.25


def numbers_in(text):
    """Every number written in a text, as (value, decimals shown)."""
    out = []
    for m in re.findall(NUMBER, str(text)):
        m = m.rstrip(",")
        decimals = len(m.split(".")[1]) if "." in m else 0
        out.append((float(m.replace(",", "")), decimals))
    return out


def unsourced(text, sources):
    """Numbers in text that no source holds, at the decimals quoted."""
    pool = [v for s in sources for v, _ in numbers_in(
        s if isinstance(s, str) else json.dumps(s, default=str))]
    return [q for q, d in numbers_in(text)
            if not any(abs(round(v, d) - q) < 1e-9 for v in pool)]


def sources_of(task, steps):
    """What an agent was given or shown: the task, its own tool arguments
    and every tool result."""
    return [task] + [x for s in steps for x in (s.get("args"), s.get("result"))
                     if x is not None]


answer = hunt[-1]["answer"]
claim = answer["failure"]
rerun = TEST_TOOLS[claim["tool"]](**claim["args"])
seen = [s["result"] for s in hunt
        if s.get("tool") == claim["tool"] and s.get("args") == claim["args"]]
print("claimed failure:", claim)
print("rerun:", rerun)
print("same as the agent saw:", bool(seen) and seen[0] == rerun)
print("a band goes negative:",
      rerun.get("first month a band is negative") is not None)
print("numbers in the report without a source:",
      unsourced(answer["report"], sources_of(HUNT, hunt)) or "none")

The rerun gives the same result the agent saw, a band does go negative, and every number in the report has a source. The report is therefore true to the tools. It also names the fix: restore a limit on treatments. Whether to restore the `MIN`, or protect each band as chapter 12's Try it 5 did, is the modeller's decision; the agent found the failure, not the right structure.

Two things made the hunt work. The tools ran tests that could fail, and the plausible ranges told the agent where failure would matter: the rewrite breaks inside them, not only at absurd values.

**Try it 2.**

- *Check.* In which month does a band first go negative with `referrals_0` = 3,000? Find it in the agent's steps, not by running the model.
- *Apply.* The tools are plain functions. Call `summary` on the rewrite with the `MIN` restored, at both of the agent's failing settings, and confirm that the failure is gone.
- *Extend.* Write a check of the agent's method, not its numbers: did it call `halve_dt` on exactly the setting it reports as its failure?

<details><summary>Worked answer</summary>

*Check.* Month 34.5, read from the step that called `extreme` with those arguments.

*Apply.* With the `MIN` restored, `capacity_0` = 5,000 ends with 2,690 patients and a lowest band of 7, and `referrals_0` = 3,000 ends with 2,017 and a lowest band of 5. No band goes negative.

*Extend.* The check in the next cell finds a `halve_dt` call whose settings match the claimed failure: True. A report that rules out the time step should be backed by a step that did so; this check makes that a rule rather than a hope.

</details>

In [ ]:
fixed = variant.with_equation(
    "Treatments", "MIN(Capacity, Waiting_list / Minimum_treatment_time)")
print("check:", [s["result"]["first month a band is negative"]
                 for s in hunt if s.get("args") == {"parameter": "referrals_0",
                                                    "value": 3000}])
for setting in ({"capacity_0": 5000}, {"referrals_0": 3000}):
    print("apply:", setting, summary(fixed, setting))
ruled_out = any(s.get("tool") == "halve_dt"
                and s["args"]["settings"] == {claim["args"]["parameter"]:
                                              claim["args"]["value"]}
                for s in hunt)
print("extend: the agent ruled out the time step for its claim:", ruled_out)

## 3. Machine learning: hybrid models

*Stage task:* estimate a relationship the structure leaves open. *Kind of AI:* machine learning, here least squares on a flexible lookup table; no language model is involved. *Check:* test years the fit never saw, and a plot of the learned piece against its training range. *What stays human:* what the effect does outside the data.

A **hybrid model** keeps the parts of the structure we are sure of (stocks, flows, units, conservation of patients) and learns from data only the part we are unsure of. In the waiting list we are sure that the list is referrals in minus treatments out. We are less sure how treatments respond to pressure: when the list is long, staff run extra clinics and work faster, up to a limit. That response is an **effect**, a dimensionless multiplier, the Effect of a Dimensionless Input (`effectFunction`) of chapter 7, after Hines, *Molecules of Structure* [@hines_molecules].

To test the method where we know the truth, we build a "true" world with a known effect table, generate ten years of monthly data from it, and then pretend we do not know the table. Unlike the published data, suppose the hospital's own records give monthly referrals and capacity.

In [ ]:
TRUE_EFFECT = "TABLE(Pressure, 0.5, 0.85, 1, 1, 1.5, 1.1, 2, 1.15, 3, 1.17)"


def pressure_model(effect_eq):
    return lb.Model.from_spec({
        "name": "Waiting list with pressure", "timeUnit": "month",
        "T": 120, "dt": 0.25,
        "stocks": [{"id": "wl", "name": "Waiting list", "init": "30000",
                    "units": "patients", "nonNegative": True}],
        "flows": [
            {"id": "r", "name": "Referrals", "to": "wl", "eq": "4000",
             "units": "patients/month"},
            {"id": "t", "name": "Treatments", "from": "wl",
             "eq": "MIN(Capacity * Effect_of_pressure, "
                   "Waiting_list / Minimum_treatment_time)",
             "units": "patients/month"}],
        "vars": [
            {"id": "c", "name": "Capacity", "eq": "4000",
             "units": "patients/month"},
            {"id": "rl", "name": "Reference list", "eq": "40000",
             "units": "patients"},
            {"id": "p", "name": "Pressure",
             "eq": "Waiting_list / Reference_list", "units": "1"},
            {"id": "e", "name": "Effect of pressure", "eq": effect_eq,
             "units": "1"},
            {"id": "m", "name": "Minimum treatment time", "eq": "0.5",
             "units": "month"}]})


months = np.arange(121.0)
referrals = pd.Series(4000 + 900 * np.sin(2 * np.pi * months / 60)
                      + np.cumsum(rng.normal(0, 40, 121)), index=months)
# inputs= replaces the Referrals equation with this monthly series
truth = pressure_model(TRUE_EFFECT).run(inputs={"Referrals": referrals})
observed = truth.loc[months, "Waiting list"].to_numpy() \
    * (1 + rng.normal(0, 0.005, 121))
print(truth.loc[[0.0, 30.0, 60.0, 120.0],
                ["Waiting list", "Pressure", "Treatments"]].round(2))

In this "true" world referrals wander, and the list with them: from 30,000 patients at the start to 45,590 at month 30, then down to 21,443 at month 60 and 5,685 at the end. Pressure, the list divided by a reference of 40,000 patients, ranges from 0.75 to 1.14 and down to 0.14 in the rows shown. That variation is what makes the effect learnable: a list that never moved would tell us nothing about how treatments respond to it.

The learning step uses the structure. Because the list changes only through its flows, the treatments in month $t$ are the referrals minus the change in the list, and dividing by capacity gives the effect:

$$\widehat{\text{Treatments}}_t = \text{Referrals}_t - (W_{t+1} - W_t),\qquad \widehat{\text{Effect}}_t = \widehat{\text{Treatments}}_t / \text{Capacity}.$$

We fit a lookup table to these points. A table that draws straight lines between points is linear in its $y$ values, so ordinary least squares can fit it with a **hat function** basis: each point of the table gets a column that is 1 at its own $x$ and falls to 0 at its neighbours. We place the table's points across the range of pressure seen in the first eight years, and keep the last two for testing.

In [ ]:
train = months[:-1] < 96
pressure = observed[:-1] / 40000
effect_hat = (referrals.to_numpy()[:-1] - np.diff(observed)) / 4000
lo, hi = pressure[train].min(), pressure[train].max()
knots = np.round(np.linspace(lo, hi, 5), 3)


# np.clip holds x between the first and last knot. np.interp(x, knots, ys)
# joins the points (knot, y) with straight lines; with ys a row of the
# identity matrix np.eye, 1 at knot j and 0 elsewhere, that is hat j
def hats(x, knots):
    x = np.clip(x, knots[0], knots[-1])
    return np.column_stack([np.interp(x, knots, np.eye(len(knots))[j])
                            for j in range(len(knots))])


y_knots = np.linalg.lstsq(hats(pressure[train], knots), effect_hat[train],
                          rcond=None)[0]
learned = "TABLE(Pressure, " + ", ".join(
    f"{x:.3f}, {y:.4f}" for x, y in zip(knots, y_knots)) + ")"
print(f"Pressure seen in training: {lo:.2f} to {hi:.2f}; in the test "
      f"years: {pressure[~train].min():.2f} to {pressure[~train].max():.2f}"
      f" ({(pressure[~train] < lo).mean():.0%} of them below {lo:.2f})")
print("Learned:", learned)
hybrid = pressure_model(learned).run(inputs={"Referrals": referrals})
err = hybrid.loc[months, "Waiting list"].to_numpy() - observed
print(f"Hybrid model, mean absolute error: training "
      f"{np.abs(err[:96]).mean():.0f}, test {np.abs(err[96:]).mean():.0f}"
      " patients")

In [ ]:
#| fig-alt: "Left: effect of pressure against pressure, with dots inferred from data, the true curve and the learned table; a shaded band marks the training range. Right: the observed waiting list and the hybrid model over ten years, with a vertical line at month 96."
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
xg = np.linspace(0.1, 2, 200)
true_pts = [0.5, 0.85, 1, 1, 1.5, 1.1, 2, 1.15, 3, 1.17]
ax1.plot(pressure[train], effect_hat[train], "o", ms=3,
         label="inferred from data")
# [0::2] takes every second item from the first (the x values), [1::2] the y
ax1.plot(xg, np.interp(xg, true_pts[0::2], true_pts[1::2]),
         label="true effect")
ax1.plot(xg, np.interp(xg, knots, y_knots), label="learned table")
ax1.axvspan(lo, hi, alpha=0.1, color="grey")    # shade from x = lo to x = hi
ax1.set(title="The learned effect", xlabel="Pressure (list / 40,000)",
        ylabel="Effect of pressure (dimensionless)")
ax1.legend()
ax2.plot(months, observed / 1000, label="observed")
ax2.plot(months, hybrid.loc[months, "Waiting list"] / 1000,
         label="hybrid model")
ax2.axvline(96, color="grey", lw=1)
ax2.set(title="The hybrid model against the data", xlabel="month",
        ylabel="Waiting list (thousand patients)")
ax2.legend()
plt.tight_layout()
plt.show()

*Figure: left, the effect of pressure on treatments inferred from the data (dots), the true effect and the learned table; the shaded band is the range of pressure seen in training. Right, the hybrid model with the learned table against the data; the vertical line marks the start of the test years.*

The learned table follows the true effect closely over the training range of pressure, 0.53 to 1.16; at a pressure of 1.0 it gives 0.990, against a true 1.0. Put back into the stock-and-flow structure, it reproduces the list with a mean absolute error of 262 patients in the training years and 222 in the two test years, which it never saw.

Look at the test years before celebrating. Their pressure runs from 0.15 to 0.65, and 79% of the test months lie below anything seen in training. Below its first point the table is flat, by the convention of `TABLE`, and the true effect happens to be flat there too, at 0.85. The good test score is partly luck. A hybrid model is honest about this because the learned piece is small, has units and a meaning, and can be plotted and argued over. Richer learned pieces exist. **Neural ordinary differential equations** let a neural network stand in for part or all of the rate equations, trained through the simulator [@chen2018]; **physics-informed neural networks** train a network to fit the data while obeying known equations [@raissi2019]. Both go further than a table, and both inherit its weakness outside the data.

**Try it 3.**

- *Check.* What is the learned effect at a pressure of 1.0? How close is it to the true value?
- *Apply.* Fit the table with only the first four years of data. How wide is the range of pressure seen, and how does the test error change?
- *Extend.* Replace the table by a straight line, `a + b * Pressure`, fitted by least squares to the same points. How does it do on the test years, and what does it say at a pressure of 3?

<details><summary>Worked answer</summary>

*Check.* 0.990, within 1% of the true value of 1.0.

*Apply.* The first four years show pressure only from 0.72 to 1.16, so the table's lowest point sits at 0.72 and it must guess below that. The test error rises from 222 to 972 patients.

*Extend.* The line, effect = 0.689 + 0.311 x pressure, misses the test years by 1,356 patients on average, because it carries on falling below the training range where the true effect is flat. At a pressure of 3 it says 1.62, against a true 1.17: it promises that staff go on working ever faster as the list grows. A table flat beyond its ends is a cautious guess; a line is a confident wrong one. Neither replaces asking clinicians what happens at extreme pressure.

</details>

In [ ]:
print(f"Learned effect at 1.0: {np.interp(1.0, knots, y_knots):.3f} "
      "(true 1.0)")
early = months[:-1] < 48
lo4, hi4 = pressure[early].min(), pressure[early].max()
k4 = np.round(np.linspace(lo4, hi4, 5), 3)
y4 = np.linalg.lstsq(hats(pressure[early], k4), effect_hat[early],
                     rcond=None)[0]
t4 = "TABLE(Pressure, " + ", ".join(f"{x}, {y:.4f}" for x, y in
                                     zip(k4, y4)) + ")"
h4 = pressure_model(t4).run(inputs={"Referrals": referrals})
e4 = h4.loc[months, "Waiting list"].to_numpy() - observed
print(f"Four years: pressure {lo4:.2f} to {hi4:.2f}, "
      f"test error {np.abs(e4[96:]).mean():.0f}")
a, b = np.linalg.lstsq(np.column_stack([np.ones(train.sum()),
                                        pressure[train]]),
                       effect_hat[train], rcond=None)[0]
hl = pressure_model(f"{a:.4f} + {b:.4f} * Pressure").run(
    inputs={"Referrals": referrals})
el = hl.loc[months, "Waiting list"].to_numpy() - observed
print(f"Line: effect = {a:.3f} + {b:.3f} x pressure; test error "
      f"{np.abs(el[96:]).mean():.0f}; at pressure 3 it says "
      f"{a + 3 * b:.2f} (true 1.17)")

## 4. Machine learning: a forecasting contest across a policy change

*Stage task:* test whether the model forecasts better than a model of the data alone. *Kind of AI:* machine learning, an autoregression fitted by least squares. *Check:* years held back from both models, and a "no change" benchmark. *What stays human:* knowing that a policy changed, and putting it in the model.

Now the data scientist's question. In January 2022 the hospital group launched a waiting-list initiative that bought about 10% more treatment capacity. Train two models on the data up to December 2021, and forecast 2022 and 2023, which we hold back.

- The **pure data model** is an **autoregression**: next month's change in the list is a weighted sum of the last $p$ monthly changes, plus a constant, fitted by least squares. It knows nothing about stocks or policies.
- The **system dynamics model** is the one-stock case: `Waiting list` with `Referrals` and `Treatments = MIN(Capacity, Waiting_list / Minimum_treatment_time)`, referrals and capacity growing at constant rates, and the 2020 fall in activity (capacity down to about a third in March 2020, back over nine months) as a known input. We calibrate four constants by least squares with chapter 13's Nelder-Mead search, inside a wide box that keeps capacity and referrals between 3,000 and 5,000 patients/month and the growth rates between minus 5% and 10% a year.

The data are synthetic, shaped like published Irish waiting-list counts; see chapter 13.

In [ ]:
y = wl["total"].to_numpy(dtype=float)
t_all = np.arange(len(y), dtype=float)           # months since Jan 2014
TRAIN_END = 96                                   # Jan 2022 is month 96

sd_model = lb.Model.from_spec({
    "name": "Waiting list case", "timeUnit": "month", "T": 119, "dt": 0.25,
    "stocks": [{"id": "wl", "name": "Waiting list", "init": str(y[0]),
                "units": "patients", "nonNegative": True}],
    "flows": [
        {"id": "r", "name": "Referrals", "to": "wl",
         "eq": "Referrals_base * (1 + Referral_growth) ^ (TIME / 12)",
         "units": "patients/month"},
        {"id": "t", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
         "units": "patients/month"}],
    "vars": [
        {"id": "rb", "name": "Referrals base", "eq": "4000",
         "units": "patients/month"},
        {"id": "rg", "name": "Referral growth", "eq": "0.03",
         "units": "fraction/year"},
        {"id": "cb", "name": "Capacity base", "eq": "3950",
         "units": "patients/month"},
        {"id": "cg", "name": "Capacity growth", "eq": "0.02",
         "units": "fraction/year"},
        {"id": "pan", "name": "Pandemic effect",
         "eq": "TABLE(TIME, 73.9, 1, 74, 0.35, 83, 1)", "units": "1"},
        {"id": "ini", "name": "Initiative size", "eq": "0.10",
         "units": "fraction"},
        {"id": "cap", "name": "Capacity",
         "eq": "Capacity_base * (1 + Capacity_growth) ^ (TIME / 12)"
               " * Pandemic_effect * (1 + Initiative_size * STEP(1, 96))",
         "units": "patients/month"},
        {"id": "mtt", "name": "Minimum treatment time", "eq": "0.5",
         "units": "month"}]})
SD_KEYS = ["referrals_base", "referral_growth", "capacity_base",
           "capacity_growth"]


def sd_run(theta, T=119, **kw):
    d = sd_model.run(T=T, include=["Waiting list"],
                     **dict(zip(SD_KEYS, theta)), **kw)
    return d.loc[np.arange(T + 1.0), "Waiting list"].to_numpy()


def nelder_mead(f, x0, step, max_evals=3000, tol=1e-9):
    """Minimise f from x0 with chapter 13's Nelder-Mead simplex method.

    Returns the best point, its value and the number of evaluations.
    """
    n = len(x0)
    X = np.array([x0] + [x0 + np.eye(n)[i] * step[i] for i in range(n)],
                 dtype=float)
    F = np.array([f(x) for x in X])
    evals = n + 1
    while evals < max_evals:
        order = np.argsort(F)
        X, F = X[order], F[order]                 # best first, worst last
        if F[-1] - F[0] <= tol * (abs(F[0]) + 1e-12):
            break
        centre = X[:-1].mean(axis=0)              # of all but the worst
        xr = centre + (centre - X[-1])            # reflect the worst
        fr = f(xr)
        evals += 1
        if fr < F[0]:                             # very good: go further
            xe = centre + 2 * (centre - X[-1])
            fe = f(xe)
            evals += 1
            X[-1], F[-1] = (xe, fe) if fe < fr else (xr, fr)
        elif fr < F[-2]:                          # good enough: keep it
            X[-1], F[-1] = xr, fr
        else:                                     # poor: pull back in
            xc = centre + 0.5 * ((xr if fr < F[-1] else X[-1]) - centre)
            fc = f(xc)
            evals += 1
            if fc < min(fr, F[-1]):
                X[-1], F[-1] = xc, fc
            else:                                 # shrink towards the best
                X[1:] = X[0] + 0.5 * (X[1:] - X[0])
                F[1:] = [f(x) for x in X[1:]]
                evals += n
    i = int(np.argmin(F))
    return X[i], F[i], evals


BOX = np.array([[3000, -0.05, 3000, -0.05], [5000, 0.10, 5000, 0.10]])


def calibrate(end):
    """Least squares on months 0 to end - 1, inside a wide box."""
    def sse(th):
        if np.any(th < BOX[0]) or np.any(th > BOX[1]):
            return np.inf
        return ((sd_run(th, T=end - 1) - y[:end]) ** 2).sum()
    return nelder_mead(sse, np.array([3800, 0.02, 3800, 0.01]),
                       [200, 0.01, 200, 0.01], 1500)[0]


theta = calibrate(TRAIN_END)
print({k: round(float(v), 4) for k, v in zip(SD_KEYS, theta)})

The fit gives referrals of 3,553 patients/month growing 6.4% a year and capacity of 3,508 patients/month growing 5.4% a year. These need not be the true values: the stock alone pins down mostly the difference between referrals and capacity, and the 2020 fall in activity is what pins down the level of capacity (@sec-calibration). For forecasting the list, the difference is what counts.

The autoregression is fitted to monthly changes, and forecasts are built by feeding each predicted change back in. We fit orders $p$ of 1, 2, 3, 6 and 12 months.

In [ ]:
def ar_fit(x, p):
    # a column of ones, then x shifted by 1, 2, ..., p months: the lags
    rows = np.column_stack([np.ones(len(x) - p)]
                           + [x[p - k - 1:len(x) - k - 1] for k in range(p)])
    return np.linalg.lstsq(rows, x[p:], rcond=None)[0]


def ar_forecast(x, coef, h):
    hist = list(x)
    for _ in range(h):
        hist.append(coef[0] + sum(c * hist[-k - 1]
                                  for k, c in enumerate(coef[1:])))
    return np.array(hist[len(x):])


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


test = slice(TRAIN_END, len(y))         # y[test] is y[TRAIN_END:len(y)]
h = len(y) - TRAIN_END
dy = np.diff(y[:TRAIN_END])
forecasts = {}
for p in (1, 2, 3, 6, 12):
    forecasts[f"autoregression, p = {p}"] = \
        y[TRAIN_END - 1] + np.cumsum(ar_forecast(dy, ar_fit(dy, p), h))
forecasts["no change (last value)"] = np.full(h, y[TRAIN_END - 1])
forecasts["system dynamics, policy ignored"] = \
    sd_run(theta, initiative_size=0)[test]
forecasts["system dynamics, with the initiative"] = sd_run(theta)[test]
score = pd.Series({k: rmse(v, y[test]) for k, v in forecasts.items()})
print("Error over 2022 and 2023 (RMSE, patients):")
print(score.round(0).sort_values().to_string())
print("End of 2023, autoregression p = 2:",
      round(forecasts["autoregression, p = 2"][-1]), "| data:", round(y[-1]))

In [ ]:
#| fig-alt: "The waiting list from 2014 to 2023 with three forecasts from the end of 2021: an autoregression that keeps climbing, a system dynamics model without the initiative that climbs further, and one with the initiative that levels off with the data."
fig, ax = plt.subplots(figsize=(9, 4.5))
dates = wl["month"]
ax.plot(dates, y / 1000, label="data", lw=2.5)
for k in ("autoregression, p = 2", "system dynamics, policy ignored",
          "system dynamics, with the initiative"):
    ax.plot(dates[test], forecasts[k] / 1000, label=k)
ax.axvline(dates[TRAIN_END], color="grey", lw=1)
ax.set(title="Forecasting 2022 and 2023 from data up to 2021",
       xlabel="month", ylabel="Waiting list (thousand patients)")
ax.legend(loc="upper left")
plt.show()

*Figure: the waiting list (data) and three forecasts made at the end of 2021 (vertical line). The autoregression extends the climb it learned; the system dynamics model with the initiative bends with the data.*

Over 2022 and 2023 the system dynamics model told about the initiative has the smallest error, 872 patients. Every autoregression extends the post-pandemic climb and misses by 3,966 to 4,920 patients; the best, with $p$ = 2, ends 2023 at 76,990 against an actual 69,927. The system dynamics model without the initiative is worst of all, 6,686: structure alone does not help if the policy is left out. What wins is structure plus knowledge of the policy, and only a structural model has a place to put that knowledge.

**In quiet times.** Repeat the contest when no policy changed: train on 2014 to 2017 and forecast 2018 and 2019.

In [ ]:
QUIET = 48                                         # Jan 2018
theta_q = calibrate(QUIET)
sd_q = sd_run(theta_q)
dq = np.diff(y[:QUIET])
win = slice(QUIET, QUIET + 24)
ar_q = y[QUIET - 1] + np.cumsum(ar_forecast(dq, ar_fit(dq, 1), 24))
print(f"Two years ahead: system dynamics {rmse(sd_q[win], y[win]):.0f}, "
      f"autoregression p = 1 {rmse(ar_q, y[win]):.0f}")
one_ar, one_sd = [], []
coef = ar_fit(dq, 1)
for t in range(QUIET, QUIET + 24):
    one_ar.append(y[t - 1] + ar_forecast(np.diff(y[:t]), coef, 1)[0])
    one_sd.append(y[t - 1] + sd_q[t] - sd_q[t - 1])
print(f"One month ahead: system dynamics {rmse(one_sd, y[win]):.0f}, "
      f"autoregression p = 1 {rmse(one_ar, y[win]):.0f}, "
      f"no change {rmse(y[QUIET - 1:QUIET + 23], y[win]):.0f}")
print("Quiet-time calibration:",
      {k: round(float(v), 3) for k, v in zip(SD_KEYS, theta_q)})

In quiet times the system dynamics model still does better here: over two years it misses by 381 patients and the autoregression by 1,331; one month ahead it misses by 149 against 177, and both beat "no change" (280). But look at the quiet-time calibration. Capacity sits on the edge of the box at 3,000 patients/month, and the growth rates come out at 8.0% and 7.2% a year, outside chapter 12's plausible ranges. Without the 2020 shock the data pin down only the difference, so a model can forecast well with parameters that mean nothing. A good fit is not a correct model.

The lesson is not that one family of models is better. A data model learns the patterns in its training data and is good at continuing them; it has no way to represent a change it has never seen. A structural model represents the mechanism through which a policy acts, so it can be told about the policy; that is why it is the right tool for "what if we act?". When nothing changes and the horizon is short, the data model is a strong, cheap benchmark, and every system dynamics forecast should be compared with one [@hyndman2021].

**Try it 4.**

- *Check.* Why does the "no change" forecast do so well over 2022 and 2023?
- *Apply.* Suppose the initiative had been 20%, not 10%. Rerun the system dynamics forecast with `initiative_size=0.2` and compare its end-2023 list with the 10% forecast. What would the autoregression say?
- *Extend.* Give the autoregression a fair chance: add an input column that is 1 from January 2022 (a policy dummy) to its regression. Can it learn the effect of the initiative from the training data? Why not?

<details><summary>Worked answer</summary>

*Check.* Because the list barely moved: it rose by only 1,375 patients over the two years, because the initiative happened to bring treatments close to referrals. "No change" was right by coincidence of policy, and it would have said the same for any initiative, or none.

*Apply.* With a 20% initiative the system dynamics model ends 2023 at 55,835 patients instead of 68,791. The autoregression says 76,990 in both cases: it has no input through which the size of the initiative could act.

*Extend.* The dummy is zero in every training month (0 months have it switched on before 2022), so its column carries no information and its coefficient cannot be estimated. A data model can learn the effect of a policy only from past episodes of that policy. A structural model can reason about a new policy from how the system works.

</details>

In [ ]:
print("Change in the list during 2022-2023 (data):", y[-1] - y[TRAIN_END - 1])
s10, s20 = sd_run(theta), sd_run(theta, initiative_size=0.2)
print(f"End of 2023: 10% initiative {s10[-1]:.0f}, 20% initiative "
      f"{s20[-1]:.0f}, autoregression p = 2 "
      f"{forecasts['autoregression, p = 2'][-1]:.0f} in both cases")
dummy = (t_all[1:TRAIN_END] >= 96).astype(float)    # True/False as 1.0/0.0
print("Months with the dummy switched on in the training data:",
      int(dummy.sum()))

## 5. Agent: a calibration assistant

*Stage task:* decide which parameters the data should set. *Kind of AI:* an agent with two tools, one that fits a chosen set of parameters and one that scores the fit on a validation year. *Check:* the modeller scores the agent's choice on 2022 and 2023, which no tool can read, and a log shows how far into the data every tool looked. *What stays human:* which parameters may be freed at all, the held-out test, and the decision to accept the fit.

Chapter 13 showed that the list alone cannot identify every parameter: referrals and capacity trade off along a ridge, and a fit can be excellent with parameters that mean nothing. So the choice of which parameters to free is a judgement, and a natural job for an assistant: try sets, compare, and argue for one. The danger is just as clear. An assistant told to fit well will free everything (the Failure gallery shows it).

We use chapter 13's one-stock model, with its seven parameters, plausible ranges and error scale. The parameters not freed stay at the values reported by the hospital: referrals and capacity from chapter 13's first guess, and the shock and the initiative as reported. Data are read only through `observed`, which logs how far each read goes, so code can later prove which months a tool saw. `fitted` remembers each fit (`lru_cache` keeps the result of a call with the same arguments), so the agent and the checks share one search per set.

In [ ]:
def var(name, eq, units):
    return {"id": lb.name_key(name), "name": name, "eq": str(eq),
            "units": units}


CAPACITY = ("Capacity_0 * EXP(Capacity_growth * TIME / Months_per_year)"
            " * Shock_effect * (1 + STEP(Initiative_boost, Initiative_start))")
SHOCK = ("1 - STEP(1, Shock_start) * (1 - Activity_at_worst)"
         " * MAX(0, 1 - (TIME - Shock_start) / Recovery_time)")
one_stock = lb.Model.from_spec({
    "name": "Waiting list", "timeUnit": "month", "T": 119, "dt": 0.25,
    "stocks": [{"id": "wl", "name": "Waiting list",
                "init": str(wl.total[0]), "units": "patients",
                "nonNegative": True}],
    "flows": [
        {"id": "ref", "name": "Referrals", "to": "wl", "units":
         "patients/month", "eq":
         "Referrals_0 * EXP(Referral_growth * TIME / Months_per_year)"},
        {"id": "tr", "name": "Treatments", "from": "wl", "units":
         "patients/month", "eq":
         "MIN(Capacity, Waiting_list / Minimum_treatment_time)"}],
    "vars": [var("Capacity", CAPACITY, "patients/month"),
             var("Shock effect", SHOCK, "dmnl"),
             var("Referrals 0", 4000, "patients/month"),
             var("Referral growth", 0.03, "1/year"),
             var("Capacity 0", 3950, "patients/month"),
             var("Capacity growth", 0.02, "1/year"),
             var("Activity at worst", 0.35, "dmnl"),
             var("Recovery time", 9, "month"),
             var("Shock start", 73, "month"),
             var("Initiative boost", 0.10, "dmnl"),
             var("Initiative start", 95, "month"),
             var("Minimum treatment time", 0.5, "month"),
             var("Months per year", 12, "month/year")]})
REPORTED = {k: one_stock.params[k] for k in [
    "referrals_0", "referral_growth", "capacity_0", "capacity_growth",
    "activity_at_worst", "recovery_time", "initiative_boost"]}
RANGES = {"referrals_0": (3000, 5000), "referral_growth": (0.0, 0.05),
          "capacity_0": (3000, 5000), "capacity_growth": (-0.02, 0.05),
          "activity_at_worst": (0.2, 0.6), "recovery_time": (3, 18),
          "initiative_boost": (0.0, 0.2)}
obs = wl.total.to_numpy(dtype=float)
SIGMA = float(np.diff(obs).std())       # chapter 13's error scale (patients)
HOLD_OUT = 96                           # 2022 and 2023 stay unseen
read = []                               # how far each read went (months)


def observed(end):
    """The data for months 0 to end - 1. Every read is logged."""
    read.append(end)
    return obs[:end]


def fit_months(free, end):
    """Fit the free parameters to months 0 to end - 1; the rest reported."""
    lo = np.array([RANGES[k][0] for k in free])
    hi = np.array([RANGES[k][1] for k in free])

    def payoff(z):
        if np.any(z < lo) or np.any(z > hi):
            return np.inf
        sim = one_stock.run(T=end - 1, include=["Waiting list"],
                            warn_clips=False, **{**REPORTED,
                                                 **dict(zip(free, z))})
        err = sim["Waiting list"].to_numpy()[::4] - observed(end)
        return float(((err / SIGMA) ** 2).sum())

    x0 = np.array([REPORTED[k] for k in free], dtype=float)
    z, _, _ = nelder_mead(payoff, x0, 0.1 * (hi - lo), 2000)
    z, _, _ = nelder_mead(payoff, z, 0.02 * (hi - lo), 1000)
    return {**REPORTED, **dict(zip(free, z))}


@lru_cache(maxsize=None)                # a tuple of names is the key
def fitted(free, end):
    params = fit_months(list(free), end)
    run = one_stock.run(include=["Waiting list"], **params)
    return params, run["Waiting list"].to_numpy()[::4]


print(f"sigma {SIGMA:.0f} patients; parameters {list(RANGES)}")

In [ ]:
def fit(free: list) -> dict:
    """Fit the named parameters to 2014 to 2021 by least squares; the
    others stay at their reported values. Returns the training RMSE
    (patients), the estimates and any estimate on a bound of its range."""
    params, sim = fitted(tuple(free), HOLD_OUT)
    est = {k: float(f"{params[k]:.4g}") for k in free}
    edge = [k for k in free if min(abs(params[k] - b) for b in RANGES[k])
            < 0.005 * (RANGES[k][1] - RANGES[k][0])]
    return {"free": list(free), "estimates": est, "on a bound": edge,
            "training RMSE 2014-2021": round(rmse(sim[:HOLD_OUT],
                                                  observed(HOLD_OUT)))}


def validate(free: list) -> dict:
    """Fit the named parameters to 2014 to 2020 only, and score that fit
    on 2021, a year it did not see. Returns the validation RMSE
    (patients)."""
    _, sim = fitted(tuple(free), 84)
    return {"free": list(free),
            "validation RMSE 2021": round(rmse(sim[84:96],
                                               observed(96)[84:]))}


CAL_TOOLS = {"fit": fit, "validate": validate}
print(fit(["referrals_0", "capacity_0"]))

`fit` fits 2014 to 2021, the training years. `validate` refits on 2014 to 2020 and scores 2021, a year that fit did not see. Neither reads 2022 or 2023. The task tells the agent the rules of evidence: free as few parameters as the evidence supports, and treat an estimate on a bound, or a better training fit with a worse validation score, as a reason not to free one.

In [ ]:
CALIBRATE = f"""Calibrate chapter 13's one-stock waiting-list model to the
monthly list, 2014 to 2021. You choose which parameters to free; the
others stay at their reported values {REPORTED}. Parameters and plausible
ranges: {RANGES}. The data cannot identify every parameter, so free as few
as the evidence supports: an estimate on a bound, or a better training fit
with a worse validation score, is a reason not to free a parameter. The
years 2022 and 2023 are held out; you cannot see them. Finish with an
answer that is a JSON object: {{"free": [the parameters you chose],
"reasons": three sentences that quote only numbers the tools returned}}."""
SYSTEM_CAL = "You are a careful calibration assistant for a modeller."
read.clear()                            # watch what the agent's tools read
cal = ai.agent(CALIBRATE, CAL_TOOLS, system=SYSTEM_CAL, replay=REPLAY,
               max_steps=12)
ai.show(cal, width=110)
agent_read = max(read)                  # the furthest month they read

The agent worked in pairs of calls, fit then validate. Freeing only the two levels gave a training RMSE of 508 patients and a validation RMSE of 919. Freeing the growth rates as well improved training to 472, but `referral_growth` landed on its bound at 0.05 and validation worsened to 976, so the agent kept the growth rates at their reported values. Freeing `recovery_time` with the levels improved both: training 459 and validation 445, with a recovery time of 10.34 months. Freeing the depth of the shock too improved training to 452 and worsened validation to 730, so it stopped there.

Now the modeller's check, on the years the agent never saw. `held_out` scores each choice on 2022 and 2023; it is the only function that reads them, and it is not a tool.

In [ ]:
def held_out(free):
    """The modeller's check: RMSE on 2022 and 2023, which no tool showed."""
    _, sim = fitted(tuple(free), HOLD_OUT)
    return round(rmse(sim[HOLD_OUT:], observed(len(obs))[HOLD_OUT:]))


chosen = cal[-1]["answer"]["free"]
CH13 = ["referrals_0", "referral_growth", "capacity_0", "capacity_growth"]
rows = []
for label, free in [("agent's choice", chosen), ("chapter 13's choice", CH13),
                    ("referrals_0 and capacity_0 only",
                     ["referrals_0", "capacity_0"])]:
    rows.append({"choice": label, "free": len(free),
                 "training": fit(free)["training RMSE 2014-2021"],
                 "validation 2021": validate(free)["validation RMSE 2021"],
                 "held out 2022-23": held_out(free)})
print(pd.DataFrame(rows).to_string(index=False))
print("months of data the agent's tools read:", agent_read,
      "| held-out years unseen:", agent_read <= HOLD_OUT)
print("numbers in the reasons without a source:",
      unsourced(cal[-1]["answer"]["reasons"], sources_of(CALIBRATE, cal))
      or "none")

In [ ]:
#| fig-alt: "The waiting list from 2014 to 2023 with two calibrated models fitted to 2014 to 2021; after the vertical line at January 2022 the agent's choice stays closer to the data than chapter 13's choice."
fig, ax = plt.subplots()
ax.plot(wl["month"], obs / 1000, color="black", label="data")
for label, free in [("agent's choice", chosen),
                    ("chapter 13's choice", CH13)]:
    ax.plot(wl["month"], fitted(tuple(free), HOLD_OUT)[1] / 1000,
            label=f"{label}: {', '.join(free)}")
ax.axvline(wl["month"][HOLD_OUT], color="grey", lw=1)
ax.set(title="Calibrated to 2014-2021, judged on 2022-2023",
       xlabel="month", ylabel="Waiting list (thousand patients)")
ax.legend(fontsize=8, loc="upper left")
plt.show()

*Figure: the waiting list and two models calibrated to 2014 to 2021: the agent's choice and chapter 13's choice. After the vertical line (January 2022) both are forecasts.*

On 2022 and 2023 the agent's choice misses by 1,281 patients, against 2,420 for chapter 13's choice of the four flow parameters and 1,602 for the two levels alone. The log shows that the agent's tools read only the first 96 months, to the end of 2021, so the held-out years were never seen. And every number in its reasons has a source.

The agent did not find the truth. It found the set that the evidence in the training years supports best, by rules we gave it. "Chapter 13's choice" is the set it used for its out-of-sample test: the referral and capacity parameters, with the events as reported. That comparison does not make chapter 13 wrong, since it used the set to show identifiability, not to forecast. What the agent added was speed, and a record of every set it tried. What stays human is the list of candidate parameters, the rules, the held-out test, and the judgement that a recovery time of 10.34 months, against a reported 9, is believable.

**Try it 5.**

- *Check.* Which parameters did the agent free, and what is their held-out RMSE against chapter 13's choice?
- *Apply.* Call `fit` and `validate` yourself on a set the agent did not try: the two levels, `capacity_growth` and `recovery_time`. Would the agent's rules accept it?
- *Extend.* One validation year is thin evidence. Write `validate2`, the mean of two folds (fit to 2014 to 2019 and score 2020; fit to 2014 to 2020 and score 2021), a small **rolling-origin** validation. Does it rank the sets the same way?

<details><summary>Worked answer</summary>

*Check.* `referrals_0`, `capacity_0` and `recovery_time`, with a held-out RMSE of 1,281 patients against 2,420.

*Apply.* Training improves to 451, with no estimate on a bound, but validation worsens to 751 from the agent's 445. The rules reject it: a better training fit with a worse validation score.

*Extend.* The two-fold scores are 1,626 for the two levels, 1,116 for chapter 13's four, 830 for the agent's three and 4,601 with the depth of the shock added. The agent's set is still the best; the depth of the shock does far worse, because a fit that ends in 2019 has never seen a shock and cannot estimate its depth. More folds give a steadier ranking at the price of more fits [@hyndman2021].

</details>

In [ ]:
print("check:", chosen, held_out(chosen), "against", held_out(CH13))
other = ["referrals_0", "capacity_0", "capacity_growth", "recovery_time"]
print("apply:", fit(other), validate(other))


def validate2(free):
    """Two folds: fit to 2014-2019, score 2020; fit to 2014-2020, score
    2021. The mean of the two RMSEs (patients)."""
    a = fitted(tuple(free), 72)[1]
    b = fitted(tuple(free), 84)[1]
    return round((rmse(a[72:84], observed(84)[72:])
                  + rmse(b[84:96], observed(96)[84:])) / 2)


for free in (["referrals_0", "capacity_0"], CH13, chosen,
             chosen + ["activity_at_worst"]):
    print("extend:", free, validate2(free))

## 6. A caution: overfitting, leakage and explainability

*Stage task:* keep a fitted or learned model honest. *Kind of AI:* machine learning, the autoregression of section 4, and the calibrated model. *Check:* errors on held-out years, with the split fixed before any model is chosen. *What stays human:* the split, and the one-sentence account of what a learned part does.

Three failures account for most disappointing results when machine learning meets system dynamics [@james2021].

**Overfitting.** A flexible model can match the training data ever more closely while forecasting worse. The autoregression shows it: as the order rises, the in-sample error falls, but the forecast error over 2022 and 2023 does not follow.

In [ ]:
P0 = 24                     # every order is fitted to the same months
rows = []
for p in (1, 2, 3, 6, 12, 18, 24):
    X = np.column_stack([np.ones(len(dy) - P0)]
                        + [dy[P0 - k - 1:len(dy) - k - 1] for k in range(p)])
    coef = np.linalg.lstsq(X, dy[P0:], rcond=None)[0]
    in_sample = rmse(X @ coef, dy[P0:])
    fc = y[TRAIN_END - 1] + np.cumsum(ar_forecast(dy, coef, h))
    rows.append({"order p": p, "coefficients": p + 1,
                 "training error (change/month)": in_sample,
                 "forecast error 2022-23 (patients)": rmse(fc, y[test])})
print(pd.DataFrame(rows).round(0).to_string(index=False))

Every order is fitted to the same months (from 2016 on, after the first 24 changes), so the errors differ from those of section 4. The training error falls steadily, from 414 to 369 patients/month, as the order rises from 1 to 24. The forecast error does not follow: it stays between 5,465 and 5,871 patients up to order 18, then jumps to 20,022 at order 24, the model with the best fit.

**Leakage** is information from the test period that slips into training. It makes a method look better than it is, and it is easy to do by accident: calibrate on the whole series and then "test" on its last two years; or choose among models by their test score, which turns the test set into training data. Here is the first kind.

In [ ]:
theta_all = calibrate(len(y))
print(f"Calibrated on data up to 2021: test error "
      f"{rmse(sd_run(theta)[test], y[test]):.0f}")
print(f"Calibrated on all data (leak): test error "
      f"{rmse(sd_run(theta_all)[test], y[test]):.0f}")

Calibrated on all the data, the "test" error falls from 872 to 373 patients. Nothing has improved: the model has simply seen the answers. In a report this number would overstate accuracy by more than half.

**Explainability.** A decision maker who asks "why does the list keep growing?" can be answered from the system dynamics model in its own terms: referrals grow faster than capacity, and the list is the accumulated difference. An autoregression's coefficients have no such reading, and a neural network's even less. Hybrid models sit in between: keep the learned parts small, give them units and a meaning, plot them, and test them under extreme conditions as in @sec-testing. A checklist before trusting any learned component:

1. Was it tested on data it never saw, split by time, not at random?
2. Was anything about the test period used in training, scaling or model choice?
3. Does it behave sensibly outside the range of its training data?
4. Can you say in one sentence what it does, in the units of the problem?

**Try it 6.**

- *Check.* In the overfitting table, which order has the lowest training error, and what is its forecast error?
- *Apply.* Make the second kind of leakage: choose the autoregression order by the 2022 to 2023 error, then report that error as "the" forecast accuracy. Why is the number now biased?
- *Extend.* Split the training data into 2014 to 2019 for fitting and 2020 to 2021 for choosing the order (a validation set). Which order is chosen, and what is its honest error on 2022 to 2023?

<details><summary>Worked answer</summary>

*Check.* Order 24, with a training error of 369 patients/month and a forecast error of 20,022 patients, the worst in the table.

*Apply.* Choosing on the test set picks order 18, with an error of 5,465. That number is biased downwards because the order was chosen to make it small: it is no longer a test, and a new two years of data would almost certainly show a larger error.

*Extend.* With 2020 and 2021 as a validation period, order 24 is chosen and its honest error on 2022 and 2023 is 20,022 patients. The validation years were the pandemic, unlike anything before or after, and they rewarded a model with a long memory of the shock. A validation set must resemble the future you want to forecast; and an honest number is sometimes an ugly one.

</details>

In [ ]:
ov = pd.DataFrame(rows).set_index("order p")
print("Lowest training error at p =",
      ov["training error (change/month)"].idxmin())
print("Best order chosen on the test set:",
      ov["forecast error 2022-23 (patients)"].idxmin(), "with",
      round(ov["forecast error 2022-23 (patients)"].min()))
VAL = 72                                           # Jan 2020
dv = np.diff(y[:VAL])
val_err = {}
for p in (1, 2, 3, 6, 12, 18, 24):
    fc = y[VAL - 1] + np.cumsum(ar_forecast(dv, ar_fit(dv, p),
                                            TRAIN_END - VAL))
    val_err[p] = rmse(fc, y[VAL:TRAIN_END])
p_val = min(val_err, key=val_err.get)       # the key with the smallest value
honest = y[TRAIN_END - 1] + np.cumsum(ar_forecast(dy, ar_fit(dy, p_val), h))
print("Order chosen on 2020-21:", p_val, "| honest error on 2022-23:",
      round(rmse(honest, y[test])))

## Failure gallery

Each failure below is shown by a replayed reply or by code, and caught by a check. The replies in this gallery were written for the book to show common faults; they are not a live model's output.

**1. An agent that frees parameters until the fit looks good.** Give the calibration assistant of section 5 the same tools, but a different instruction: get the training fit as low as you can. We wrote its reply to do what that instruction rewards.

In [ ]:
GREEDY = CALIBRATE.split("The data cannot")[0] + """Get the training fit
as low as you can. Finish with an answer that is a JSON object:
{"free": [the parameters you chose], "reasons": one sentence}."""
greedy = ai.agent(GREEDY, CAL_TOOLS, system=SYSTEM_CAL, replay=REPLAY,
                  max_steps=12)
ai.show(greedy, width=110)
g_free = greedy[-1]["answer"]["free"]
print(f"\ngreedy: {len(g_free)} free, training "
      f"{fit(g_free)['training RMSE 2014-2021']}, held out "
      f"{held_out(g_free)} | agent of section 5: training "
      f"{fit(chosen)['training RMSE 2014-2021']}, held out "
      f"{held_out(chosen)}")
print("bounds hit:", fit(g_free)["on a bound"], "| initiative_boost:",
      fit(g_free)["estimates"]["initiative_boost"])
no_boost = [k for k in g_free if k != "initiative_boost"]
print("the same without initiative_boost: held out", held_out(no_boost))

The agent added parameters while the training error fell: 508, 472, 442 and 442 patients, and chose all seven. It never called `validate`, and it ignored the warning that `referral_growth` is on its bound. On the held-out years its choice misses by 11,180 patients, against 1,281 for the section 5 choice. The initiative boost acts in only one training month, December 2021, and the fit pushed it to 0.1987, near the top of its range: that is fitting the noise. Without it, the other six free parameters miss the held-out years by 3,147. *Caught by:* the held-out years, which no tool could read.

**2. A confident wrong explanation of a test failure.** Ask a reasoning model why the rewrite fails `never_negative` at the top of the capacity range. We wrote this reply to be wrong in a way that sounds expert: it blames the time step.

In [ ]:
failed = run_test(variant, {"check": "never_negative",
                            "settings": {"capacity_0": 5000}})
WHY = f"""In our waiting-list model this test failed: with capacity_0 = 5000
(the top of its plausible range), never_negative gave {failed}.
Here are the equations:

{variant.equations()}

Explain the cause in two sentences and say what fix would make the test
pass."""
explanation = ai.ask(WHY, system=SYSTEM_TESTER, reasoning=True,
                     replay=REPLAY)
print(explanation)

In [ ]:
at = halve_dt({"capacity_0": 5000})
print(at)
print("claim 'dt 0.0625 keeps the bands non-negative' holds:",
      at["0.0625"]["first month a band is negative"] is None)
rk4 = variant.run(capacity_0=5000, method="rk4", include=BANDS)
print(f"RK4: lowest band {rk4.min().min():,.0f}")
print("restore the MIN:", run_test(fixed, {"check": "never_negative",
                                           "settings": {"capacity_0": 5000}}))

The claim is specific, so it can be tested: at `dt` = 0.0625 the bands should stay non-negative. They do not. A band is negative from month 33.25 at every time step, RK4 drives the lowest band to minus 10,051,811 patients, and restoring the `MIN` makes the test pass with a lowest band of 7. The equations needed a change after all. *Caught by:* rerunning the experiment the explanation implies [@ji2023hallucination].

**3. Leakage through a tool.** A helpful tool that also reports the error on 2022 and 2023 lets an agent choose by the test years, and then the "held-out" score is no longer held out. The guard is the read log of section 5: run a tool and see whether it read any month after 2021.

In [ ]:
def fit_and_score(free: list) -> dict:
    """A leaky tool: the training fit and the error on 2022 and 2023."""
    return {**fit(free), "held-out RMSE": held_out(free)}


def leak_guard(tool, *args):
    """Run a tool; True if it read any month of the held-out years."""
    n = len(read)
    tool(*args)
    return max(read[n:], default=0) > HOLD_OUT


for tool in (fit, validate, fit_and_score):
    print(f"{tool.__name__}: reads held-out months:",
          leak_guard(tool, ["referrals_0", "capacity_0"]))

`fit` and `validate` pass; the leaky `fit_and_score` is caught before any agent uses it. *Caught by:* logging every read of the data, and refusing tools that read the held-out years.

**4. A test plan that tests only what passes.** Ask for a plan "that shows the model is ready" for the board, and the plan will try to show it. We wrote this reply to show the pattern: six tests, five at the base settings and one extreme that any model passes.

In [ ]:
PASSING = f"""Here is a system dynamics model of an outpatient waiting list,
monthly from January 2014 to December 2023.

{case.equations()}

We present it to the hospital board next week. Propose a short test plan
that shows the model is ready. Each test uses one of these checks, with
"settings" as parameter values:
{CHECKS}

Return a JSON list; each test is an object with keys "name", "check",
"settings" (a dict, may be empty) and "why" (one sentence). At most 8."""
plan_b = ai.ask_json(PASSING, system="You are a helpful modelling "
                     "assistant.", replay=REPLAY)
power = pd.DataFrame({
    "plan": ["section 1 (could fail)", "for the board (shows ready)"],
    "tests": [len(plan), len(plan_b)],
    "fail on the case model": [
        (run_plan(case, p).result == "FAIL").sum() for p in (plan, plan_b)],
    "fail on the rewrite": [
        (run_plan(variant, p).result == "FAIL").sum()
        for p in (plan, plan_b)]})
print(power.to_string(index=False))
print(run_plan(variant, plan_b).to_string(index=False))

The board plan passes on the case model, and it passes on the rewrite too: zero failures in six tests, though the rewrite drives the list negative inside the plausible range. Section 1's plan fails the rewrite six times. A plan that cannot fail on a known fault has no power, however reassuring its passes. *Caught by:* mutation testing, running the plan on a model with a planted fault.

**5. A number nobody computed.** A report can quote real numbers and slip in one of its own. Here is a sentence written for the book with one invented share.

In [ ]:
report = ("The rewrite fails at capacity_0 = 5000: a band first goes "
          "negative at month 33.25 and the list reaches -225 patients, "
          "about 7% of capacity.")
print("unsourced:", unsourced(report, sources_of(HUNT, hunt)))
print(f"the true share: {-rerun['lowest list'] / claim['args']['value']:.1%}")

The check flags 7: no tool returned it, and the true share, 225 against 5,000, would be 4.5%. Month 33.25 and minus 225 have sources and pass. *Caught by:* `unsourced`, on every report, every time.

**Try it 7.**

- *Check.* Which tests of the board plan push any input outside the history?
- *Apply.* List the tests of section 1's plan that fail on the rewrite, with their evidence.
- *Extend.* Run `unsourced` on the greedy agent's reasons. Why does a bad choice pass the number check?

<details><summary>Worked answer</summary>

*Check.* Only one, the run with no capacity, and it cannot fail on the rewrite: with no capacity nothing is treated, so the missing limit never matters.

*Apply.* Six fail: no referrals (the list ends at 45,878.4 with a lowest band of minus 397.8, because treatments go on after the list is empty), capacity at the top of its range, ten times the capacity, urgent first, the shortest treatment time, and halving `dt` with three times the capacity (a 507.40% change).

*Extend.* Nothing is flagged: 442 is a number a tool returned. The check proves that a report is true to the tools, not that the choice is wise. Judgement needs its own checks, here the held-out years.

</details>

In [ ]:
pushed = [t for t in plan_b if t["settings"]]
print("check: tests outside the history:", [t["name"] for t in pushed])
on_rewrite = run_plan(variant, plan)
print("apply: section 1 tests that catch the rewrite:")
print(on_rewrite[on_rewrite.result == "FAIL"][["test", "detail"]]
      .to_string(index=False))
print("extend: unsourced numbers in the greedy agent's reasons:",
      unsourced(greedy[-1]["answer"]["reasons"], sources_of(GREEDY, greedy))
      or "none")

## Check it

Before an AI step in testing or calibration is kept:

1. **Run, do not trust.** Every test a model proposes is run by code, and every failure an agent claims is rerun.
2. **Source every number.** Run `unsourced` on every report; a number without a source is removed or recomputed.
3. **Guard the held-out years.** Log every read of the data, and check that no tool an agent can call reads the test period.
4. **Measure a plan's power.** Run it on a model with a planted fault; it must fail.
5. **Keep the record.** Every reply has its key, its date and how it was made.

In [ ]:
reports = {"testing agent": (answer["report"], sources_of(HUNT, hunt)),
           "calibration agent": (cal[-1]["answer"]["reasons"],
                                 sources_of(CALIBRATE, cal))}
for who, (text, src) in reports.items():
    print(f"{who}: every number sourced:", not unsourced(text, src))
print("claimed failure reruns the same:", rerun == seen[0])
print("held-out years unseen by the agent's tools:", agent_read <= HOLD_OUT)
print("the test plan fails on a planted fault:",
      (run_plan(variant, plan).result == "FAIL").any())
print("every reply was written for the book or recorded live:",
      sorted({r["how"] for r in records.values()}))

## Running case

The waiting-list model is now tested and calibrated with AI help, each step recorded and checked. A reasoning model proposed 12 tests; code ran 10, all passed, and two went to a person, including the boundary question chapter 12 left open. A knockout showed that the limit on treatments never acts in the history, and a testing agent then found that a rewrite without it breaks inside the plausible range, a claim the rerun confirmed. A calibration agent freed the two levels and the recovery time; on 2022 and 2023, which no tool could read, that choice misses by 1,281 patients. The tested and calibrated model goes on to @sec-model-to-decision, and @sec-ai-decide brings AI to the decision stage.

## Tell the decision maker

"We tested the model with the help of AI tools, and checked every claim they made by rerunning the model. The tests found a version of the model that would have given impossible answers for exactly the capacity question you are asking; it is not the one we use. We calibrated the model on data to 2021 and checked it on 2022 and 2023, which it had not seen: it was out by about 1,300 patients on a list of about 70,000. A forecast built only from past data would have missed the effect of your initiative; ours did not, because the model can be told about it."

## Going further

- **Mutation testing.** Plant many small faults automatically (a changed constant, a swapped variable, a dropped `MIN`) and score a test sheet by the share it catches. The typo of Try it 1 shows why a sheet of passes is not enough.
- **Rolling-origin validation.** Move the split forward a year at a time and refit each time [@hyndman2021]; it gives several validation errors instead of one, and shows whether estimates drift as data accumulate.
- **Structured tools.** Model services offer native tool calling and JSON output; the book's plain text protocol keeps every step visible and replayable, and the checks are the same with either.
- **Calibration as testing.** @oliva2003 treats calibration as a set of structural tests; an agent that fits partial models, where flows are known, would follow that advice.
- **Learned pieces.** Universal differential equations embed small networks inside known equations, the idea of section 3 taken further [@chen2018; @raissi2019].
- **Cautions on language models in simulation.** @akhavan2024 discusses how such models should, and should not, be used in simulation modelling.

## Mini-project

Take the AI steps of this chapter to chapter 12's wider model, in which referrals respond to the wait.

1. Run section 1's plan on it, add a test of the bands so that the typo of Try it 1 is caught, and report the power of the plan on three planted faults of your own.
2. Give the calibration agent `wait_elasticity` as a candidate parameter (write its replies with `ai.author`, or run live), and score its choice on 2022 and 2023.
3. Say whether the data identify the elasticity, using a profile as in chapter 13.
4. Write three sentences for the team: what the tests found, which parameters the data support, and how far to trust the forecast.

## Further reading

On validity and the tests behind this chapter see @barlas1996, and on calibration as a test of structure @oliva2003; @martinezmoyano2013 sets out good practice in modelling, including testing. On language models in simulation modelling see @akhavan2024; on agents that reason and act with tools, @yao2022react; on reasoning step by step, @wei2022cot; and on why fluent text can be false, @ji2023hallucination. For forecasting and its validation, @hyndman2021; for overfitting and validation sets, @james2021. For learned pieces inside differential equations, @chen2018 and @raissi2019.